# EV Spare Parts Inventory & Forecasting Analysis

## Project overview
We supply spare parts to several EV service centers from a central warehouse. Using weekly
historical demand and inventory data, this notebook investigates demand patterns, inventory
performance, and forecasting accuracy, then connects forecast quality to real business cost.

## Central question
**Can a forecasting method with low statistical error still create high inventory cost?**

A method can look good on paper (low MSE/MAE/MAPE) while still being expensive in practice,
if its errors are systematically biased in one direction (e.g. consistently under-forecasting
a part with a high stockout cost, or over-forecasting a part with a high holding cost). Every
section below builds toward answering this question, comparing methods both on statistical
error and on the business cost they would have generated.

## Data
Three weekly CSVs live in `data/`, joined on (`week_start`, `part_id`):
- **parts.csv** — one row per part: identity, cost, price, supplier, lead time, cost
  parameters used later for reorder logic.
- **demand.csv** — weekly units requested per part, by service center region and vehicle type.
- **inventory.csv** — weekly stock on hand and replenishment, recorded *before* that week's
  demand is fulfilled.

## Analysis plan
1. **Data audit** (this section) — shapes, dtypes, missing values, duplicates, date gaps,
   negative values.
2. Merge + derived variables, with a sanity check that demand reconciles with fulfilled + unmet.
3. Exploratory analysis of demand, stockouts, fill rate, and cost exposure.
4. Forecasting with several methods (naive, moving average, exponential smoothing,
   Holt-Winters, Croston's for intermittent demand).
5. Dual evaluation: statistical error vs. business cost, per method.
6. Reorder logic and the final `reorder_recommendations.csv` output.


## Phase 1: Data audit

Before doing any analysis we need to know whether the data can be trusted at face value.
For each of the three source files we check:
- **Shape and dtypes** — do columns have the types we expect (dates as dates, IDs as strings,
  quantities as numbers)?
- **Missing values** — are there gaps that would break a merge or a calculation?
- **Duplicates** — is `(part_id, week_start)` unique in `demand` and `inventory`, and is
  `part_id` unique in `parts`? A duplicate key would silently corrupt the merge in Phase 2.
- **Date gaps** — does every part have a reading for every week in the observation window?
- **Negative values** — quantities, costs, and lead times should never be negative.


In [2]:
import pandas as pd
import numpy as np

# Load the three source tables. week_start is parsed as a date up front since
# every downstream step (merging, gap-checking, forecasting) depends on it.
parts = pd.read_csv("../data/parts.csv")
demand = pd.read_csv("../data/demand.csv", parse_dates=["week_start"])
inventory = pd.read_csv("../data/inventory.csv", parse_dates=["week_start"])

print("parts:", parts.shape)
print("demand:", demand.shape)
print("inventory:", inventory.shape)

parts.head()


parts: (24, 10)
demand: (1872, 6)
inventory: (1872, 4)


,part_id,part_name,category,unit_cost,unit_price,supplier,lead_time_days,demand_trend,holding_cost_per_unit_week,stockout_cost_per_unit
0,P001,Battery Module A,Battery System,820,1150,VoltCore GmbH,21,Increasing,4.92,847.50
1,P002,Battery Cooling Plate,Battery System,145,240,ThermoEV Supply,14,Increasing,0.87,203.00
2,P003,Battery Management Sensor,Battery System,95,165,SensorTech EU,14,Stable,0.57,144.25
3,P004,Charging Port Assembly,Charging System,210,340,ChargeLink AG,21,Increasing,1.26,249.00
4,P005,AC Charging Cable,Cables and Connectors,75,135,CableWorks,7,Stable,0.45,93.75


In [3]:
# Check dtypes: week_start should be datetime64, IDs/categoricals should be
# object (string), and quantities/costs should be int64 or float64.
print("=== parts dtypes ===")
print(parts.dtypes)
print("\n=== demand dtypes ===")
print(demand.dtypes)
print("\n=== inventory dtypes ===")
print(inventory.dtypes)


=== parts dtypes ===
part_id                        object
part_name                      object
category                       object
unit_cost                       int64
unit_price                      int64
supplier                       object
lead_time_days                  int64
demand_trend                   object
holding_cost_per_unit_week    float64
stockout_cost_per_unit        float64
dtype: object

=== demand dtypes ===
week_start               datetime64[ns]
part_id                          object
units_demanded                    int64
service_center_region            object
vehicle_type                     object
service_campaign                  int64
dtype: object

=== inventory dtypes ===
week_start       datetime64[ns]
part_id                  object
stock_on_hand             int64
replenishment             int64
dtype: object


In [11]:
# Check for missing values in each table.
print("=== parts missing values ===")
print(parts.isna().sum())
print("\n=== demand missing values ===")
print(demand.isna().sum())
print("\n=== inventory missing values ===")
print(inventory.isna().sum())


=== parts missing values ===
part_id                       0
part_name                     0
category                      0
unit_cost                     0
unit_price                    0
supplier                      0
lead_time_days                0
demand_trend                  0
holding_cost_per_unit_week    0
stockout_cost_per_unit        0
dtype: int64

=== demand missing values ===
week_start               0
part_id                  0
units_demanded           0
service_center_region    0
vehicle_type             0
service_campaign         0
dtype: int64

=== inventory missing values ===
week_start       0
part_id          0
stock_on_hand    0
replenishment    0
dtype: int64


In [6]:
# Check for duplicate keys: part_id must be unique in parts.csv, and
# (part_id, week_start) must be unique in demand and inventory.
print("duplicate part_id in parts:", parts.duplicated(subset=["part_id"]).sum())
print("duplicate (part_id, week_start) in demand:", demand.duplicated(subset=["part_id", "week_start"]).sum())
print("duplicate (part_id, week_start) in inventory:", inventory.duplicated(subset=["part_id", "week_start"]).sum())

# Also confirm the same set of parts appears in all three tables.
parts_ids = set(parts["part_id"])
demand_ids = set(demand["part_id"])
inventory_ids = set(inventory["part_id"])
print("\nparts not in demand:", parts_ids - demand_ids)
print("demand parts not in parts.csv:", demand_ids - parts_ids)
print("inventory parts not in parts.csv:", inventory_ids - parts_ids)


duplicate part_id in parts: 0
duplicate (part_id, week_start) in demand: 0
duplicate (part_id, week_start) in inventory: 0

parts not in demand: set()
demand parts not in parts.csv: set()
inventory parts not in parts.csv: set()


In [9]:
print(inventory.head())
print(f"\n now the demand table: \n {demand.head()}")

  week_start part_id  stock_on_hand  replenishment
0 2024-01-01    P001             25              0
1 2024-01-08    P001             22              0
2 2024-01-15    P001             19              0
3 2024-01-22    P001             15             30
4 2024-01-29    P001             41              0

 now the demand table: 
   week_start part_id  units_demanded service_center_region vehicle_type  \
0 2024-01-01    P001               3                  West   Compact EV   
1 2024-01-08    P001               3                  West   Compact EV   
2 2024-01-15    P001               4               Central   Compact EV   
3 2024-01-22    P001               4                 North   Compact EV   
4 2024-01-29    P001               4                  West   Compact EV   

   service_campaign  
0                 0  
1                 0  
2                 1  
3                 0  
4                 0  


In [12]:
# Unique part counts and the observation window (date range) for each table.
print("unique part_ids:", parts["part_id"].nunique())
print("\ndemand date range:", demand["week_start"].min(), "to", demand["week_start"].max())
print("inventory date range:", inventory["week_start"].min(), "to", inventory["week_start"].max())

n_weeks = demand["week_start"].nunique()
print("\nunique weeks in demand:", n_weeks)
print("parts x weeks:", parts["part_id"].nunique() * n_weeks, "vs demand rows:", len(demand))


unique part_ids: 24

demand date range: 2024-01-01 00:00:00 to 2025-06-23 00:00:00
inventory date range: 2024-01-01 00:00:00 to 2025-06-23 00:00:00

unique weeks in demand: 78
parts x weeks: 1872 vs demand rows: 1872


In [13]:
# Explicit per-part date-gap check: build the full expected set of weeks,
# then compare it against each part's actual weeks in demand and inventory.
expected_weeks = pd.date_range(demand["week_start"].min(), demand["week_start"].max(), freq="7D")

def find_gaps(df, name):
    gaps = {}
    for pid, group in df.groupby("part_id"):
        missing = set(expected_weeks) - set(group["week_start"])
        if missing:
            gaps[pid] = sorted(missing)
    if gaps:
        print(f"{name}: parts with missing weeks -> {gaps}")
    else:
        print(f"{name}: no missing weeks for any part ({len(expected_weeks)} expected weeks each)")

find_gaps(demand, "demand")
find_gaps(inventory, "inventory")


demand: no missing weeks for any part (78 expected weeks each)
inventory: no missing weeks for any part (78 expected weeks each)


In [14]:
# Negative-value check: none of these quantities/costs should ever be negative.
numeric_checks = {
    "parts.unit_cost": parts["unit_cost"],
    "parts.unit_price": parts["unit_price"],
    "parts.lead_time_days": parts["lead_time_days"],
    "parts.holding_cost_per_unit_week": parts["holding_cost_per_unit_week"],
    "parts.stockout_cost_per_unit": parts["stockout_cost_per_unit"],
    "demand.units_demanded": demand["units_demanded"],
    "inventory.stock_on_hand": inventory["stock_on_hand"],
    "inventory.replenishment": inventory["replenishment"],
}

for label, series in numeric_checks.items():
    n_negative = (series < 0).sum()
    print(f"{label}: {n_negative} negative values")


parts.unit_cost: 0 negative values
parts.unit_price: 0 negative values
parts.lead_time_days: 0 negative values
parts.holding_cost_per_unit_week: 0 negative values
parts.stockout_cost_per_unit: 0 negative values
demand.units_demanded: 0 negative values
inventory.stock_on_hand: 0 negative values
inventory.replenishment: 0 negative values


### Phase 1 summary — GO

The audit found no issues:
- **Shape**: 24 parts, 1872 weekly rows each in `demand` and `inventory` (24 parts x 78 weeks,
  exactly as expected — a complete rectangular panel).
- **Dtypes**: all columns typed as expected (`week_start` as datetime, IDs/categories as
  strings, quantities/costs as int/float). No coercion needed.
- **Missing values**: none, in any of the three tables.
- **Duplicate keys**: `part_id` is unique in `parts`; `(part_id, week_start)` is unique in
  both `demand` and `inventory`. All three tables reference the same 24 part_ids — no orphans
  on either side. No duplicate-resolution step is needed before merging.
- **Date gaps**: every part has all 78 weeks (2024-01-01 to 2025-06-23) in both `demand` and
  `inventory` — no missing weeks to worry about being misread as zero-demand.
- **Negative values**: none found across any cost, quantity, or lead-time column.

The data is clean enough to merge as-is. Next: merge `demand` and `inventory` on
`(week_start, part_id)` and compute the derived variables (`available_inventory`,
`units_fulfilled`, `unmet_demand`, `ending_stock`, `stockout`).


## Phase 2: Merge demand + inventory

We merge `demand` and `inventory` on `(week_start, part_id)` to get one row per part per week
with both demand and stock information. Since Phase 1 confirmed both tables have exactly the
same 1872 `(part_id, week_start)` combinations with no duplicates, this merge should produce
exactly 1872 rows — we assert that explicitly to catch any silent fan-out.


In [15]:
# Merge on (week_start, part_id). Using how="left" with demand as the base
# table is a deliberate choice: demand is the table we ultimately care about
# explaining, and Phase 1 already confirmed inventory has a matching row for
# every demand row, so left vs. inner makes no difference here in practice.
df = demand.merge(inventory, on=["week_start", "part_id"], how="left")

# Assert no silent row duplication happened during the merge.
assert len(df) == len(demand), f"Merge changed row count: {len(demand)} -> {len(df)}"
print(f"Merge OK: {len(df)} rows, {df['part_id'].nunique()} parts")

df.head()


Merge OK: 1872 rows, 24 parts


,week_start,part_id,units_demanded,service_center_region,vehicle_type,service_campaign,stock_on_hand,replenishment
0,2024-01-01,P001,3,West,Compact EV,0,25,0
1,2024-01-08,P001,3,West,Compact EV,0,22,0
2,2024-01-15,P001,4,Central,Compact EV,1,19,0
3,2024-01-22,P001,4,North,Compact EV,0,15,30
4,2024-01-29,P001,4,West,Compact EV,0,41,0


### Derived variables

Using the exact formulas specified for this project:

- `available_inventory = stock_on_hand + replenishment` — total units available to fulfill
  demand that week (stock already on hand plus whatever arrived from suppliers).
- `units_fulfilled = min(available_inventory, units_demanded)` — we can't fulfill more than
  either what was demanded or what was available.
- `unmet_demand = max(units_demanded - available_inventory, 0)` — demand left unsatisfied;
  clipped at 0 so a surplus week doesn't produce a negative "unmet" value.
- `ending_stock = max(available_inventory - units_demanded, 0)` — leftover stock after
  fulfilling demand; clipped at 0 for the same reason.
- `stockout = 1 if unmet_demand > 0 else 0` — a simple flag for whether the part ran out
  that week, used later for stockout-rate analysis.


In [16]:
# Derived variables, computed exactly as specified. We use np.minimum/np.maximum
# (not Python's min/max) because these operate element-wise across a whole column
# at once, rather than on a single pair of values.
df["available_inventory"] = df["stock_on_hand"] + df["replenishment"]
df["units_fulfilled"] = np.minimum(df["available_inventory"], df["units_demanded"])
df["unmet_demand"] = np.maximum(df["units_demanded"] - df["available_inventory"], 0)
df["ending_stock"] = np.maximum(df["available_inventory"] - df["units_demanded"], 0)
df["stockout"] = (df["unmet_demand"] > 0).astype(int)

df[["week_start", "part_id", "units_demanded", "stock_on_hand", "replenishment",
    "available_inventory", "units_fulfilled", "unmet_demand", "ending_stock", "stockout"]].head(10)


,week_start,part_id,units_demanded,stock_on_hand,replenishment,available_inventory,units_fulfilled,unmet_demand,ending_stock,stockout
0,2024-01-01,P001,3,25,0,25,3,0,22,0
1,2024-01-08,P001,3,22,0,22,3,0,19,0
2,2024-01-15,P001,4,19,0,19,4,0,15,0
3,2024-01-22,P001,4,15,30,45,4,0,41,0
4,2024-01-29,P001,4,41,0,41,4,0,37,0
5,2024-02-05,P001,5,37,0,37,5,0,32,0
6,2024-02-12,P001,2,32,0,32,2,0,30,0
7,2024-02-19,P001,3,30,0,30,3,0,27,0
8,2024-02-26,P001,5,27,0,27,5,0,22,0
9,2024-03-04,P001,1,22,20,42,1,0,41,0


### Sanity checks

Before trusting these derived columns for any downstream analysis, we verify three things
hold across the *entire* dataset, not just the few rows we eyeballed above:

1. **Reconciliation**: total `units_demanded` must equal total `units_fulfilled` + total
   `unmet_demand`. This is a direct consequence of the formulas, but computing it end-to-end
   catches any implementation mistake.
2. **No negatives**: none of the five derived columns should ever be negative (by
   construction, thanks to `min`/`max`, but we verify rather than assume).
3. **Chaining property**: for a given part, this week's `ending_stock` should equal next
   week's `stock_on_hand` — that's how the physical inventory system works, and we saw it
   hold by eye for P001 above. We check it holds for every part, every week.


In [45]:
# Check 1: total demand reconciles with fulfilled + unmet.
total_demanded = df["units_demanded"].sum()
total_fulfilled = df["units_fulfilled"].sum()
total_unmet = df["unmet_demand"].sum()
check1 = total_demanded == total_fulfilled + total_unmet
print(f"[{'PASS' if check1 else 'FAIL'}] total_demanded ({total_demanded}) == "
      f"total_fulfilled ({total_fulfilled}) + total_unmet ({total_unmet})")

# Check 2: no negatives in any derived column.
derived_cols = ["available_inventory", "units_fulfilled", "unmet_demand", "ending_stock"]
n_negative = (df[derived_cols] < 0).sum().sum()
check2 = n_negative == 0
print(f"[{'PASS' if check2 else 'FAIL'}] negative values across derived columns: {n_negative}")

# Check 3: chaining property, per part. Sort by part then week, shift ending_stock
# down by one row within each part group, and compare to that row's stock_on_hand.
df_sorted = df.sort_values(["part_id", "week_start"]).copy()
df_sorted["prev_ending_stock"] = df_sorted.groupby("part_id")["ending_stock"].shift(1)

# The first week of each part has no "previous" row, so we exclude those (NaN) rows.
chain_check = df_sorted.dropna(subset=["prev_ending_stock"])
mismatches = chain_check[chain_check["prev_ending_stock"] != chain_check["stock_on_hand"]]
check3 = len(mismatches) == 0
print(f"[{'PASS' if check3 else 'FAIL'}] chaining property holds for "
      f"{len(chain_check) - len(mismatches)}/{len(chain_check)} part-week pairs")
if len(mismatches) > 0:
    print(mismatches[["part_id", "week_start", "prev_ending_stock", "stock_on_hand"]].head())


[PASS] total_demanded (25034) == total_fulfilled (24476) + total_unmet (558)
[PASS] negative values across derived columns: 0
[PASS] chaining property holds for 1848/1848 part-week pairs


### Layer 1 summary — GO

The full reconstruction is verified end-to-end:
- Data audit found no missing values, no duplicate keys, no date gaps, and no negative values
  across all three source tables.
- The merge of `demand` and `inventory` on `(week_start, part_id)` produced exactly the
  expected 1872 rows, with no fan-out.
- The five derived variables reconcile perfectly: total demand (25,034 units) equals total
  fulfilled (24,476) plus total unmet (558) — an overall unmet-demand rate of about 2.2%.
- The physical chaining property (one week's `ending_stock` equals the next week's
  `stock_on_hand`) holds for all 1,848 part-week transitions checked.

`df` is now a trustworthy foundation for Phase 3 (exploratory analysis of demand, stockouts,
fill rate, and cost exposure) and everything that follows.


## Phase 3: Exploratory data analysis

With a clean, verified `df`, we now explore the data itself, building toward two things:
which forecasting methods are even appropriate for which parts (some parts have steady
demand, others have long stretches of zero demand — that changes which method makes sense),
and where the cost exposure actually sits (which parts are expensive to under- or over-stock).

This section covers: demand character per part, a reality check on the synthetic
`demand_trend` labels, demand by category/region, inventory KPIs (stockout rate, fill rate),
historical holding/stockout cost per part, and a cost-vs-volume segmentation. We build up a
single `part_profile` table across these steps that later phases (forecasting, reorder logic)
will reuse.

### Demand character per part

For each part we compute the mean and standard deviation of weekly demand, and the share of
weeks with zero demand, as basic descriptive statistics. The proper classification of demand
*pattern* (smooth vs. intermittent) follows in the next cells using a standard method, rather
than an ad-hoc threshold on zero-demand share alone.


In [ ]:
# Per-part demand statistics: mean, standard deviation, and share of zero-demand weeks.
part_profile = df.groupby("part_id")["units_demanded"].agg(
    mean_demand="mean",
    std_demand="std",
    n_weeks="count",
).reset_index()

zero_share = df.groupby("part_id")["units_demanded"].apply(lambda s: (s == 0).mean())
part_profile["zero_demand_share"] = part_profile["part_id"].map(zero_share)

# Attach part name and category for readability, and carry demand_trend along
# so we can compare the synthetic label against what we actually observe later.
part_profile = part_profile.merge(
    parts[["part_id", "part_name", "category", "demand_trend"]], on="part_id"
)

part_profile.sort_values("zero_demand_share", ascending=False)


### Demand pattern classification (Syntetos-Boylan)

Rather than an arbitrary threshold on zero-demand share, we use the standard method from
demand-planning literature for deciding when intermittent-demand forecasting (Croston's
method) is justified. Two measures, computed per part from its own weekly demand history:

- **ADI (Average Demand Interval)**: the average number of weeks between weeks with nonzero
  demand, computed as `total weeks / number of nonzero-demand weeks`. ADI = 1 means demand
  happens every week; ADI = 5 means demand happens roughly once every 5 weeks on average.
- **CV² (squared coefficient of variation)**: computed only over the nonzero-demand weeks,
  as `(std of nonzero demand / mean of nonzero demand)^2`. Low CV² means that when demand
  does happen, the quantity is fairly consistent; high CV² means it swings wildly.

Standard cutoffs (Syntetos & Boylan): **ADI > 1.32** flags infrequent demand, **CV² > 0.49**
flags erratic order sizes. Combining both gives four categories:

| | CV² <= 0.49 | CV² > 0.49 |
|---|---|---|
| **ADI <= 1.32** | Smooth | Erratic |
| **ADI > 1.32** | Intermittent | Lumpy |

Croston's method (Phase 4) is only justified for parts landing in Intermittent or Lumpy.


In [ ]:
def adi_cv2(units_demanded):
    """Compute ADI and CV^2 for one part's weekly demand series."""
    n_weeks = len(units_demanded)
    nonzero = units_demanded[units_demanded > 0]
    n_nonzero = len(nonzero)

    adi = n_weeks / n_nonzero
    cv2 = (nonzero.std() / nonzero.mean()) ** 2
    return pd.Series({"ADI": adi, "CV2": cv2})

# Apply the function to each part's demand series and get one row per part back.
# unstack() leaves part_id as the index, so reset_index() turns it back into a
# regular column before we merge it into part_profile.
adi_cv2_table = df.groupby("part_id")["units_demanded"].apply(adi_cv2).unstack().reset_index()
part_profile = part_profile.merge(adi_cv2_table, on="part_id")

# Classify using the standard Syntetos-Boylan cutoffs.
def classify(row):
    if row["ADI"] <= 1.32 and row["CV2"] <= 0.49:
        return "Smooth"
    elif row["ADI"] > 1.32 and row["CV2"] <= 0.49:
        return "Intermittent"
    elif row["ADI"] <= 1.32 and row["CV2"] > 0.49:
        return "Erratic"
    else:
        return "Lumpy"

part_profile["demand_pattern"] = part_profile.apply(classify, axis=1)

print(part_profile["demand_pattern"].value_counts())
part_profile.sort_values("ADI", ascending=False)[
    ["part_id", "part_name", "mean_demand", "zero_demand_share", "ADI", "CV2", "demand_pattern"]
]
